Polars supports two modes of operation: lazy and eager. The eager API executes the query/code immediately. In the lazy API, the query is only evaluated once it is collected. Deferring the execution to the last minute can have significant performance advantages and is why the lazy API is preferred in most cases. Let us demonstrate this with an example.

https://docs.pola.rs/user-guide/concepts/lazy-api/

In [1]:
import polars as pl

orders_dataset = "../data/olist_orders.csv.gz"

## eager API

In [2]:
df = pl.read_csv(orders_dataset)
df_small = df.filter(pl.col("order_status").is_in(['delivered', 'invoiced']))
df_agg = df_small.group_by("order_status").agg(pl.col("order_purchase_timestamp").min())

print(df_agg)

shape: (2, 2)
┌──────────────┬──────────────────────────┐
│ order_status ┆ order_purchase_timestamp │
│ ---          ┆ ---                      │
│ str          ┆ str                      │
╞══════════════╪══════════════════════════╡
│ delivered    ┆ 2016-09-15 12:16:38      │
│ invoiced     ┆ 2016-10-04 13:02:10      │
└──────────────┴──────────────────────────┘


## lazy API

In [3]:
q = (
    pl.scan_csv(orders_dataset)
    .filter(pl.col("order_status").is_in(['delivered', 'invoiced']))
    .group_by("order_status")
    .agg(pl.col("order_purchase_timestamp").min())
)

In [4]:
print(q.explain())

AGGREGATE[maintain_order: false]
  [col("order_purchase_timestamp").min()] BY [col("order_status")]
  FROM
  Csv SCAN [../data/olist_orders.csv.gz]
  PROJECT 2/8 COLUMNS
  SELECTION: col("order_status").is_in([["delivered", "invoiced"]])


In [5]:
df = q.collect()
df

order_status,order_purchase_timestamp
str,str
"""invoiced""","""2016-10-04 13:02:10"""
"""delivered""","""2016-09-15 12:16:38"""


-----

In [6]:
import time
import polars as pl

def eager_run():
    for i in range(10):
        df = pl.read_csv(orders_dataset)
        df_small = df.filter(pl.col("order_status").is_in(['delivered', 'invoiced']))
        df_agg = df_small.group_by("order_status").agg(pl.col("order_purchase_timestamp").min())

def lazy_run():
    for i in range(10):
        q = (
            pl.scan_csv(orders_dataset)
            .filter(pl.col("order_status").is_in(['delivered', 'invoiced']))
            .group_by("order_status")
            .agg(pl.col("order_purchase_timestamp").min())
        )
        return q.collect()

In [7]:
start = time.time()
eager_run()
print(f"eager: {(time.time() - start)*1000:.2f} ms")

start = time.time()
lazy_run()
print(f"lazy:  {(time.time() - start)*1000:.2f} ms")

eager: 579.08 ms
lazy:  81.30 ms
